<a href="https://colab.research.google.com/github/rmilde/econ_5200_applied_data_midterm_project/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Recreating main findings of Card & Krueger's DID paper:**

In [165]:
# import necessary packages

import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [168]:
# data doesn't come with column names so need to manually add those in after data is loaded
# prepare column names
col_names = [
    "SHEET","CHAIN","CO_OWNED","STATE",
    "SOUTHJ","CENTRALJ","NORTHJ","PA1","PA2","SHORE",
    "NCALLS","EMPFT","EMPPT","NMGRS","WAGE_ST","INCTIME","FIRSTINC","BONUS","PCTAFF","MEALS",
    "OPEN","HRSOPEN","PSODA","PFRY","PENTREE","NREGS","NREGS11",
    "TYPE2","STATUS2","DATE2","NCALLS2","EMPFT2","EMPPT2","NMGRS2","WAGE_ST2","INCTIME2","FIRSTIN2","SPECIAL2",
    "MEALS2","OPEN2R","HRSOPEN2","PSODA2","PFRY2","PENTREE2","NREGS2","NREGS112"
]

# load in data
public_dat_raw = pd.read_csv("data/raw/public.dat", delim_whitespace=True, header=None, names=col_names)

# display to confirm it is correct
public_dat_raw.head()

/tmp/ipykernel_174/741274669.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  public_dat_raw = pd.read_csv("public.dat", delim_whitespace=True, header=None, names=col_names)


,SHEET,CHAIN,CO_OWNED,STATE,SOUTHJ,CENTRALJ,NORTHJ,PA1,PA2,SHORE,...,FIRSTIN2,SPECIAL2,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1,2,6.50,16.50,1.03,.,0.94,4,4
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0,2,10.00,13.00,1.01,0.89,2.35,4,4
2,506,2,1,0,0,0,0,1,0,0,...,0.25,.,1,11.00,11.00,0.95,0.74,2.33,4,3
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,0.92,0.79,0.87,2,2
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,1.01,0.84,0.95,2,2


In [ ]:
# make sure number columns are numeric data
number_cols = ['EMPFT', 'EMPPT', 'NMGRS', 'WAGE_ST', 'PSODA', 'PFRY', 'PENTREE', 'EMPFT2', 'EMPPT2', 'NMGRS2', 'WAGE_ST2', 'PSODA2', 'PFRY2', 'PENTREE2']
public_dat_raw[number_cols] = public_dat_raw[number_cols].apply(pd.to_numeric, errors='coerce')

The paper states that "Full- time-equivalent [FTE] employment was calculated as the number of full-time workers [including managers] plus 0.5 times the number of part-time workers". I will use this definition to define full time workers for each store.

In [ ]:
# get number of full time workers for each store, in both the first and second interviews.
public_dat_raw['FTE'] = public_dat_raw['EMPFT'] + public_dat_raw['NMGRS'] + 0.5 * public_dat_raw['EMPPT']
public_dat_raw['FTE2'] = public_dat_raw['EMPFT2'] + public_dat_raw['NMGRS2'] + 0.5 * public_dat_raw['EMPPT2']

This is a 'Difference-in-Differences (DID)' paper, so we will exploit a natural experiment to compare treatment and control groups over time.

*   Treatment Group: NJ restaurants (Minimum wage rose from $4.25 to $5.05).
*   Control Group: PA restaurants (Minimum wage remained at $4.25).
*   Identification Assumption: Parallel Trends. In the absence of the minimum wage hike, employment trends in NJ would have been the same as in PA







In [ ]:
# For the state column, 1 if NJ; 0 if Pa
# use this to divide into control and treatment groups

treatment_nj = public_dat_raw[public_dat_raw['STATE'] == 1]
control_pa = public_dat_raw[public_dat_raw['STATE'] == 0]

### Part 1: Descriptive Table Reconstruction

In [ ]:
#Descriptive Table Reconstruction (Table 2):
# Calculate the means and standard deviations for key variables (FTE employment, starting wage, prices)
#separated by state (NJ/PA) and wave (Before/After).
#Pay close attention to the definition of "FTE" (Full-Time Equivalent)—this acts as a "unit test" for your data cleaning.

In [ ]:
# table for before and treatment (NJ)

before_cols = ['FTE', 'WAGE_ST','PSODA','PFRY','PENTREE']
treatment_nj_table_before = treatment_nj[before_cols].agg(['mean', 'std'])
treatment_nj_table_before

,FTE,WAGE_ST,PSODA,PFRY,PENTREE
mean,20.439408,4.612134,1.061508,0.941487,1.347882
std,9.106239,0.346351,0.084579,0.100778,0.646141


In [ ]:
# table for after and treatment (NJ)

after_cols = ['FTE2', 'WAGE_ST2','PSODA2','PFRY2','PENTREE2']
treatment_nj_table_after = treatment_nj[after_cols].agg(['mean', 'std'])
treatment_nj_table_after

,FTE2,WAGE_ST2,PSODA2,PFRY2,PENTREE2
mean,21.027429,5.080849,1.062154,0.960453,1.394727
std,9.293024,0.104545,0.087684,0.103518,0.660275


In [ ]:
# table for before and control (PA)

control_pa_table_before = control_pa[before_cols].agg(['mean', 'std'])
control_pa_table_before

,FTE,WAGE_ST,PSODA,PFRY,PENTREE
mean,23.331169,4.630132,0.974675,0.841948,1.215065
std,11.856283,0.351687,0.069483,0.087510,0.622889


In [ ]:
# table for after and control (PA)

control_pa_table_after = control_pa[after_cols].agg(['mean', 'std'])
control_pa_table_after

,FTE2,WAGE_ST2,PSODA2,PFRY2,PENTREE2
mean,21.165584,4.617465,0.975455,0.859863,1.185467
std,8.276732,0.357478,0.084348,0.095299,0.577961


In [ ]:
# calrify titles to prepare to combine them

treatment_nj_table_before.index = ['NJ Pre ' + i for i in treatment_nj_table_before.index]
treatment_nj_table_after.index  = ['NJ Post ' + i for i in treatment_nj_table_after.index]

control_pa_table_before.index = ['PA Pre ' + i for i in control_pa_table_before.index]
control_pa_table_after.index  = ['PA Post ' + i for i in control_pa_table_after.index]

In [ ]:
# combine into one table

fin_table = pd.concat([treatment_nj_table_before, treatment_nj_table_after, control_pa_table_before, control_pa_table_after])
fin_table

,FTE,WAGE_ST,PSODA,PFRY,PENTREE,FTE2,WAGE_ST2,PSODA2,PFRY2,PENTREE2
NJ Pre mean,20.439408,4.612134,1.061508,0.941487,1.347882,NaN,NaN,NaN,NaN,NaN
NJ Pre std,9.106239,0.346351,0.084579,0.100778,0.646141,NaN,NaN,NaN,NaN,NaN
NJ Post mean,NaN,NaN,NaN,NaN,NaN,21.027429,5.080849,1.062154,0.960453,1.394727
NJ Post std,NaN,NaN,NaN,NaN,NaN,9.293024,0.104545,0.087684,0.103518,0.660275
PA Pre mean,23.331169,4.630132,0.974675,0.841948,1.215065,NaN,NaN,NaN,NaN,NaN
PA Pre std,11.856283,0.351687,0.069483,0.087510,0.622889,NaN,NaN,NaN,NaN,NaN
PA Post mean,NaN,NaN,NaN,NaN,NaN,21.165584,4.617465,0.975455,0.859863,1.185467
PA Post std,NaN,NaN,NaN,NaN,NaN,8.276732,0.357478,0.084348,0.095299,0.577961


### Part 2: The Simple Difference

In [ ]:
#The Simple Difference:
#Calculate the "difference of means" manually.

In [ ]:
diff_before = treatment_nj_table_before.loc['NJ Pre mean', 'FTE'] - control_pa_table_before.loc['PA Pre mean', 'FTE']
diff_before

np.float64(-2.891760731480357)

In [ ]:
diff_after = treatment_nj_table_after.loc['NJ Post mean', 'FTE2'] - control_pa_table_after.loc['PA Post mean', 'FTE2']
diff_after

np.float64(-0.13815494849977483)

In [ ]:
# to see the difference in all the variables
difference = fin_table.loc['NJ Post mean'] - fin_table.loc['PA Post mean']
difference

,0
FTE,NaN
WAGE_ST,NaN
PSODA,NaN
PFRY,NaN
PENTREE,NaN
FTE2,-0.138155
WAGE_ST2,0.463384
PSODA2,0.086700
PFRY2,0.100590
PENTREE2,0.209260


### Part 3: Regression Implementation and Standard Errors and Clustering

In [ ]:
#Regression Implementation:
#Implement the regression specification using `statsmodels`.

In [ ]:
data_for_model = public_dat_raw.copy()

In [ ]:
# currently have one row for each store, but I want one row for each observation (so each store would have 2 rows)

data_for_model = data_for_model.rename(columns={'FTE': 'FTE1'})

data_for_model_long = data_for_model.melt(
    id_vars=['SHEET', 'STATE'],
    value_vars=['FTE1', 'FTE2'],
    var_name='interview',
    value_name='FTE'
)

In [ ]:
# clarify whether this is before or after the wage law in NJ took affect.

data_for_model_long['Post'] = data_for_model_long['interview'].apply(lambda x: 1 if x=='FTE2' else 0)

In [ ]:
# drop rows with missing data

data_for_model_long = data_for_model_long.dropna(subset=['FTE']).copy()

In [ ]:
# run the model and get results

model = smf.ols("FTE ~  STATE * Post", data=data_for_model_long)
results = model.fit(cov_type='cluster', cov_kwds={'groups': data_for_model_long['SHEET']})

In [ ]:
# display results

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    FTE   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.806
Date:                Mon, 09 Mar 2026   Prob (F-statistic):              0.146
Time:                        22:04:49   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.347     17.327      0.0

This matches generally what the paper found as well. The interaction term, State * Post, is 2.75.